# 00. 시작하기 — 로컬 LLM 기초 개념과 실습 환경 점검

이 노트북은 **LLM을 처음 접하는 학생**을 위한 오리엔테이션입니다. 코드는 마지막의 환경 점검 셀만 실행하면 됩니다.

**이 과정이 끝나면 할 수 있는 것**
- 내 GPU(RTX A4000 16GB)에서 LLM을 직접 실행하고, 파라미터로 출력을 조절할 수 있다 (01)
- LLM을 "서버"로 띄워 여러 프로그램이 함께 쓰게 만들 수 있다 (03~04)
- 로컬 LLM으로 구조화 출력과 RAG를 실험할 수 있다 (05, 07). 에이전트·멀티에이전트·워크플로우(06, 08, 09)는 심화 선택이다.
- 이미지 임베딩 검색까지 경험한다 (10)

> 외부 API 키나 비용은 필요 없습니다. **모든 계산이 내 컴퓨터의 GPU에서** 일어납니다.

> **6시간 핵심 과정**: 환경·연구 질문(15분) → 디코딩 실험(45분) → 로컬 모델 실행(30분) → 양자화(50분) → 구조화 출력(30분) → RAG(50분) → 멀티모달 검색(40분) → 미니 연구·결과 정리(60분). 설치·모델 다운로드 및 휴식 여유 40분을 포함합니다.
>
> **핵심 노트북**: 00 → 01 → 03 → 04 → 05 → 07 → 10. 06·08·09는 심화 선택 자료입니다.


## 1. LLM이란? — "다음 단어 맞히기"를 아주 잘하는 프로그램

LLM(Large Language Model, 대규모 언어 모델)은 **지금까지의 글을 보고 다음에 올 토큰을 예측**하는 모델입니다.

```
입력: "대한민국의 수도는"
모델: 다음 토큰 후보 →  " 서울"(92%)  " 어디"(3%)  " 세종"(1%) ...
선택: " 서울"  →  "대한민국의 수도는 서울"  → 다시 다음 토큰 예측 → ... 반복
```

| 용어 | 쉬운 설명 |
|---|---|
| **토큰(token)** | 모델이 읽고 쓰는 글자 조각. 한국어는 대략 1~2글자 = 1토큰 |
| **파라미터(parameter)** | 모델 안에 학습된 숫자들. `4B` = 40억 개, `7B` = 70억 개 |
| **학습(training)** | 대량의 글로 파라미터를 조정하는 과정 (수천 대 GPU, 수 주) |
| **추론(inference)** | 학습된 모델로 답을 생성하는 과정 ← **이 과정에서 우리가 하는 것** |
| **컨텍스트(context)** | 모델이 한 번에 볼 수 있는 토큰 수 (입력 + 출력 합계) |
| **프롬프트(prompt)** | 모델에게 주는 입력 글 |

LLM은 "사실을 검색"하는 게 아니라 "그럴듯한 다음 토큰"을 고릅니다. 그래서 **모르는 내용도 그럴듯하게 지어낼 수 있고(환각, hallucination)**, 이를 막기 위해 도구 호출(06)·RAG(07) 같은 기법을 씁니다.

## 2. 로컬 LLM vs 클라우드 API

| 비교 | 클라우드 API (예: ChatGPT API) | **로컬 LLM (이 과정)** |
|---|---|---|
| 비용 | 사용한 토큰만큼 과금 | 무료 (전기료와 GPU만 있으면 됨) |
| 데이터 | 외부 서버로 전송됨 | **내 컴퓨터 밖으로 나가지 않음** (사내 문서·개인정보에 유리) |
| 인터넷 | 필요 | 모델을 다운로드한 뒤에는 불필요 |
| 성능 | 초대형 모델이라 매우 높음 | 작은 모델(4B~7B)이라 상대적으로 낮음 → **설계로 보완** |
| 제어 | 제공되는 옵션만 사용 | 모델·양자화·서버 옵션까지 전부 직접 조절 |

> 작은 로컬 모델은 똑똑함이 부족한 대신 **명확한 프롬프트, 스키마 강제, 도구, 검색(RAG), 워크플로우**로 보완합니다. 05~09번 실습이 바로 그 방법들입니다.

## 3. GPU 메모리(VRAM) 계산법 — 모델이 내 GPU에 들어갈까?

모델을 실행하려면 파라미터 전체를 GPU 메모리(VRAM)에 올려야 합니다.

**모델 크기 ≈ 파라미터 수 × 파라미터 하나의 크기(바이트)**

| 정밀도 | 파라미터당 크기 | 4B 모델 | 7B 모델 |
|---|---|---|---|
| bf16 / fp16 (원본) | 2 바이트 | ≈ 8 GB | ≈ 14 GB |
| Q8_0 (8bit 양자화) | ≈ 1.06 바이트 | ≈ 4.5 GB | ≈ 7.5 GB |
| Q4_K_M (4bit 양자화) | ≈ 0.6 바이트 | ≈ 2.5 GB | ≈ 4.7 GB |

여기에 **KV 캐시**(대화 내용을 기억해 두는 메모리, 컨텍스트가 길수록 커짐)와 실행 오버헤드로 1~2GB가 더 필요합니다.

**RTX A4000 = 16GB VRAM**이므로 이 과정의 선택은 다음과 같습니다.
- 01: Transformers에서 모델을 직접 불러와 생성 결과와 내부 출력을 관찰
- 03~09: 4B Q8_0 또는 7B Q4_K_M(약 6GB) → 임베딩 모델과 함께 올려도 여유
- **한 번에 하나의 큰 작업만** GPU에 올립니다 (README의 VRAM 운영 규칙 참고)

> **양자화(quantization)**: 숫자를 더 적은 비트로 "반올림"해 저장하는 압축 기법입니다. 04번에서 품질이 얼마나 떨어지는지 직접 측정합니다.

## 4. 모델 이름 읽는 법

```
Qwen3.5 - 4B - Q8_0 .gguf          Qwen2.5 - 7B - Instruct - Q4_K_M .gguf
  │       │     │     └ 파일 형식      │       │      │          └ 양자화 방식(4bit)
  │       │     └ 양자화 방식(8bit)     │       │      └ 대화용으로 추가 학습된 버전
  │       └ 파라미터 40억 개            │       └ 파라미터 70억 개
  └ 모델 이름·세대 (알리바바 Qwen)       └ 모델 이름·세대
```

| 용어 | 의미 |
|---|---|
| **Base** | 다음 토큰 예측만 학습된 원본. 대화가 서툴다 |
| **Instruct / Chat** | 지시를 따르고 대화하도록 추가 학습된 버전 ← 우리가 쓰는 것 |
| **Thinking (추론 모드)** | 답하기 전에 긴 "생각 과정"을 출력하는 모드. 정확도는 오르지만 느림 → 실습에서는 **꺼 둠** |
| **VLM (Vision-Language Model)** | 이미지도 입력으로 받는 모델 (Qwen3.5는 VLM) |
| **GGUF** | llama.cpp용 모델 파일 형식. 양자화된 모델을 파일 하나로 배포 |
| **mmproj** | GGUF VLM의 "이미지 → 토큰" 변환기 파일 |
| **Hugging Face (HF)** | 모델을 공유하는 사이트. `회사/모델이름` 형식으로 내려받음 |

## 5. 연구 실험 흐름

| 단계 | 노트북 | 하는 일 |
|---|---|---|
| 모델 동작 관찰 | 01 Transformers | 모델을 직접 불러와 토큰 생성, 디코딩 설정, 출력 차이를 실험 |
| 분포·양자화 분석 | 04 양자화 | 같은 프롬프트의 토큰 확률 분포와 모델 크기를 측정 |
| 앱에 모델 연결 | 03 llama-server, 05–09 | 실험한 로컬 모델을 API로 제공하고 도구·검색 흐름에 연결 |
| 이미지 실험 | 10 SigLIP2 | 이미지·텍스트 임베딩과 검색 결과를 관찰 |

연구 질문에 맞는 노트북만 골라 실행해도 됩니다. 03번 서버는 05–09번 앱 실험이 필요할 때만 시작하세요.


## 6. 로컬 서버와 OpenAI 호환 API

`llama-server.exe`가 GGUF 모델을 불러와 `127.0.0.1:8080`에서 OpenAI 호환 API를 제공합니다. 03번에서 서버를 시작하고 05~09번 노트북이 이 주소를 사용합니다.


## 7. 앱 실습(05~09)에서 만날 핵심 용어 미리보기

| 용어 | 한 줄 설명 | 실습 |
|---|---|---|
| **시스템 프롬프트** | 모델의 역할·규칙을 정하는 지시문 ("당신은 고객지원 상담원입니다…") | 05~09 |
| **temperature** | 답변의 무작위성. 0이면 매번 같은 답, 높을수록 다양 | 01, 05~09 |
| **구조화 출력 (Structured Output)** | 자유 문장 대신 **정해진 JSON 형식**으로 답하게 강제 | 05 |
| **Pydantic** | Python에서 데이터 형식(스키마)을 정의·검증하는 라이브러리 | 05~09 |
| **도구 호출 (Tool Calling)** | 모델이 "이 함수를 이런 인자로 불러줘"라고 요청 → 우리 코드가 실행 | 03, 06 |
| **에이전트 (Agent)** | LLM + 도구 + 반복 루프. 스스로 도구를 골라 쓰며 문제를 해결 | 06~09 |
| **LangChain** | LLM 앱을 쉽게 만들게 해주는 Python 프레임워크 | 05~09 |
| **임베딩 (Embedding)** | 문장/이미지를 의미를 담은 숫자 벡터로 바꾼 것. 비슷한 의미 = 가까운 벡터 | 07, 10 |
| **RAG** | 질문과 관련된 문서를 먼저 검색해 모델에게 근거로 주는 기법 | 07 |
| **멀티에이전트** | 역할이 다른 여러 에이전트가 협업 | 08 |
| **LangGraph** | LLM 워크플로우를 노드·엣지 그래프로 설계하는 도구 | 09 |
| **파인튜닝 / LoRA** | 내 데이터로 모델을 추가 학습. LoRA는 작은 어댑터만 학습하는 경량 방식 | concept |

## 8. 과정 지도

```text
[시작] 00 환경 확인
          │
[모델 연구] 01 Transformers ─▶ 04 양자화·토큰 분포 분석
          │
[API 실험] 03 llama-server ─▶ 05 구조화 출력 ─▶ 06 에이전트
                                      └▶ 07 RAG ─▶ 08 멀티에이전트 ─▶ 09 LangGraph
          │
[확장 연구] 10 SigLIP2 이미지 검색
```

| # | 커널 | GPU 사용 | 실행 전 확인 |
|---|---|---|---|
| 01 | LLM Hands-On (Win CUDA) | Qwen3-4B (약 8–9GB) | 다른 GPU 작업 종료 |
| 03–09 | LLM Hands-On (Win CUDA) | llama-server (약 6GB), 07은 임베딩 추가 | 01 커널 종료 |
| 10 | LLM Hands-On (Win CUDA) | SigLIP2 (약 1.5GB) | VRAM 부족 시 공용 서버 종료 |


## 9. Jupyter 사용 팁

- **셀 실행**: `Shift + Enter`. 위에서부터 순서대로 실행합니다. 앞 셀에서 만든 변수를 뒤 셀이 사용합니다.
- **`NameError: name 'xxx' is not defined`**: 앞 셀을 실행하지 않았다는 뜻 → 위로 올라가 순서대로 다시 실행합니다.
- **커널 재시작 / 종료(Shutdown)**: 변수와 GPU 메모리가 모두 초기화됩니다. GPU를 비워야 할 때 Shutdown합니다.
- **`!명령어`**: 셀에서 현재 커널의 터미널 명령을 실행합니다 (예: `!nvidia-smi`). Windows 환경에서는 Bash 전용 명령을 사용하지 않습니다.
- **에러 읽는 법**: 빨간 출력의 **맨 마지막 줄**에 핵심 원인이 있습니다 (예: `CUDA out of memory` → GPU 메모리 부족).
- **TODO 셀**은 일부러 에러(`NotImplementedError`)를 냅니다. 직접 채우거나 아래 참고 답안을 보세요.

### 자주 쓰는 터미널 명령

| 명령 | 용도 |
|---|---|
| `nvidia-smi` | GPU 이름, 사용 중인 메모리, 실행 중인 프로세스 확인 |
| `nvidia-smi -l 1` | 1초마다 GPU 상태 갱신 (Ctrl+C로 종료) |
| `python local_server.py start 4b` | 공용 LLM 서버 시작 (기본 4B; `7b`도 선택 가능) |
| `python local_llm.py` | 05~09 실습 기능 전체 점검 (PASS/FAIL) |
| `python local_server.py status` | 로컬 서버 상태 확인 |
| `Get-Content runtime/llama-server-8080.log -Tail 50` | 서버 로그 마지막 50줄 |
| `python local_server.py stop` | 공용 서버 끄기 (GPU 메모리 반납) |

## 10. 실습 환경 점검

아래 셀을 실행해 ❌가 있으면 메시지의 안내대로 해결한 뒤 다시 실행하세요. (커널: `LLM Hands-On (Win CUDA)` )

In [ ]:
import sys, os, shutil, subprocess, importlib.util

def check(ok, msg, fix=""):
    print(("✅ " if ok else "❌ ") + msg + ("" if ok else f"\n    → {fix}"))

check(sys.version_info >= (3, 10), f"Python {sys.version.split()[0]}", "setup_environment.bat으로 등록한 `LLM Hands-On (Win CUDA)` 커널을 선택하세요")

try:
    import torch
    cuda = torch.cuda.is_available()
    check(cuda, f"PyTorch {torch.__version__} / CUDA 사용 가능: {cuda}",
          "setup_environment.bat을 다시 실행한 뒤 이 커널을 재시작하세요")
    if cuda:
        prop = torch.cuda.get_device_properties(0)
        free, total = torch.cuda.mem_get_info()
        print(f"    GPU: {prop.name} | VRAM {total/1024**3:.1f} GB (지금 여유 {free/1024**3:.1f} GB)"
              f" | bf16 지원: {torch.cuda.is_bf16_supported()}")
except ImportError:
    check(False, "PyTorch 미설치", "Run setup_environment.bat to install the matching CUDA PyTorch wheel")

check(shutil.which("nvidia-smi") is not None, "nvidia-smi (NVIDIA 드라이버)", "NVIDIA 드라이버를 설치하세요")

check(os.path.exists('bin/llama-server.exe'), 'bundled llama-server executable', 'Run setup_environment.bat')

for pkg in ["transformers", "accelerate", "datasets", "huggingface_hub", "openai", "langchain",
            "langchain_openai", "langgraph", "langchain_community", "langchain_classic",
            "langchain_huggingface", "sentence_transformers", "chromadb", "rank_bm25",
            "pandas", "matplotlib"]:
    check(importlib.util.find_spec(pkg) is not None, f"패키지 {pkg}", "setup_environment.bat 실행")

free_gb = shutil.disk_usage(os.path.expanduser("~")).free / 1024**3
check(free_gb > 40, f"홈 디스크 여유 {free_gb:.0f} GB", "모델 파일 저장을 위해 40GB 이상 확보 권장")
check(os.path.exists("local_llm.py") and os.path.exists("setup_environment.py"),
      "프로젝트 폴더에서 실행 중", "프로젝트 폴더의 노트북을 여세요")

## 11. (03번을 마친 뒤) 로컬 LLM에게 첫 질문하기

03번에서 서버를 띄운 뒤 아래 셀을 실행해 보세요. 이 3줄이 이후 모든 앱 실습의 출발점입니다.
서버가 아직 없으면 안내 메시지만 출력됩니다.

In [ ]:
from local_llm import get_chat_model, check_server

try:
    check_server()
    llm = get_chat_model(temperature=0)          # 로컬 서버에 연결된 LLM 객체
    answer = llm.invoke("로컬 LLM의 장점을 한 문장으로 말해줘.")
    print("🤖", answer.content)
except RuntimeError as e:
    print("아직 서버가 없습니다. 03번 노트북을 먼저 진행하세요.\n", e)

## 📖 용어집 (필요할 때 돌아와서 찾아보세요)

| 용어 | 설명 |
|---|---|
| logits | 모델이 각 토큰에 매기는 원시 점수. softmax를 거치면 확률이 됨 |
| softmax | 점수들을 합이 1인 확률로 바꾸는 함수 |
| 디코딩(decoding) | 확률분포에서 실제 다음 토큰을 고르는 방법 (greedy, sampling, beam) |
| top-k / top-p | 확률 낮은 후보를 잘라내고 남은 후보 중에서 고르는 방법 |
| chat template | 대화(역할별 메시지)를 모델이 학습한 특수 형식 텍스트로 바꾸는 규칙 |
| KV 캐시 | 이미 읽은 토큰의 계산 결과를 저장해 두는 메모리. 길게 대화할수록 커짐 |
| 배치(batch) | 여러 요청을 묶어 한 번에 처리하는 것 |
| throughput / latency | 단위 시간당 처리량 / 한 요청의 응답 대기 시간 |
| 양자화(quantization) | 파라미터를 적은 비트로 저장해 메모리를 줄이는 압축 |
| KL divergence | 두 확률분포가 얼마나 다른지 나타내는 값. 0이면 동일 |
| JSON / JSON Schema | 데이터 표기 형식 / 그 JSON의 구조 규칙을 정의하는 형식 |
| 벡터 DB (Chroma) | 임베딩 벡터를 저장하고 비슷한 벡터를 빠르게 찾는 데이터베이스 |
| BM25 | 단어가 겹치는 정도로 문서를 찾는 고전적인 키워드 검색 |
| 청크(chunk) | 긴 문서를 검색하기 좋게 자른 조각 |
| 체크포인터 / thread_id | 대화 상태를 저장하는 장치 / 대화방 번호 |
| LoRA | 원본 모델은 고정하고 작은 추가 행렬(어댑터)만 학습하는 파인튜닝 기법 |
| epoch / learning rate / loss | 데이터 전체를 한 바퀴 학습 / 한 번에 얼마나 크게 수정할지 / 정답과의 차이(낮을수록 좋음) |